# 28 · Final review on the corrected data file (16 Nov 2026)

On 16 Nov 2026 the organisers issued a corrected ball-by-ball file. In it, `extras1_runs` holds the **real extra runs**: a wide to the
boundary is 5, four byes are 4. The first file stored only a code for the type of extra, so every extra had to be counted as 1 run.
Nothing else changed: the same 867 matches and 200,999 balls, and only one player's name is written out in full.

**What the correction fixed.** These are measured with the cleaning rule in notebook 01.
- Rebuilt winning margins now match the official result in **99.0%** of matches won by runs. With the first file it was 31.7%.
- Innings totals match Cricsheet in **94.9%** of 1,560 innings (50.6% before).
- Runs charged to bowlers match Cricsheet in **94.5%** of innings (66.2% before).
- In **106 of 807 chases**, the first file said the chasing side never reached the target although it won. The corrected file has **none**.

As a result, the chase win-chance model is more accurate on matches it never saw: log-loss 0.3935 → 0.3900. Its end-of-chase
calibration error fell from 2.9% to 1.8%.

**The whole method was re-run and reviewed.** This notebook holds the tests behind each change. Every rule was fixed before running,
and ties are reported as ties.

1. **Four corrections**, each tested as a challenger:
   - Shrink each player toward the *real* average (bowlers and batters), not toward zero. The expected-value model misses byes
     and the extra runs of wides and no-balls, so the average bowler's value is not zero.
   - Fix the expected value of each ball outcome at the source (bowlers).
   - Charge the 35 dismissals off a wide to the batter.

   On the 3,998 outside matches all four are ties. On the internal checks, three are ties or slightly better; fixing the outcome
   values is worse by 4 standard errors, so it was **not** adopted. The other three were adopted. None of them changes a name in
   either top 10.
2. **The batting value.** On the corrected file the stage-by-stage search picked v1 (runs added). That choice was made in a stage
   that compares values under a rating type that is not used. Under the final setup:
   - v3 (the value submitted on 14 Nov) is better on the 52 internal checks by 3 standard errors;
   - the two tie on the outside matches (0.097 for v1 against 0.117 for v3; v1 is better in 22% of 1,000 bootstraps);
   - v3 is better on 2026.

   So **v3 is kept**. With v1, three names would change.
3. **The bowler setup.** With shrinkage toward the real average switched on, the search drifted to a different bowler setup (v3,
   consistency weight 0.5, shrinkage ×0.5, emphasis 1.0). On the outside matches it is clearly **worse** (0.245 against 0.302;
   better in only 5% of bootstraps), and it is worse on 2026. The tested setup (v1, emphasis 0.5, ×1) is kept.

   The 12 Nov decision to use shrinkage ×2 is retired: ×1 against ×2 is a tie everywhere, with the same 10 bowlers.

**Pipeline order fix.** The step that links outside matches to the supplied ones (by innings totals) now runs after the totals are rebuilt, so on the corrected file 3,998 outside matches are used. The first run today linked them with the old totals and used 3,993; the top-10 figures on outside matches moved by a few runs between the two (they are noisy), the lists did not.

**Result:** both final lists hold **the same 10 names as submitted**, in a slightly different order. Both setups are written in full
in `config.yaml`, with the evidence for each.


In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # repository root, where the small rrlib helper package lives
import rrlib.common

In [2]:
%run ./11a_settings_test_setup.ipynb

19:35:37 52 splits


## 28.1 · Four corrections, tested as challengers

<details><summary>Technical notes and the rule fixed before running</summary>

P31 - Final review (16 Nov 2026, corrected ball-by-ball file): four CORRECTIONS of known errors, each tested  
against the current method as a challenger. Nothing existing is changed by this script.  
  
Corrections (all found in the code review on the corrected file):  
  BOWL-A  shrink toward the real average: empirical Bayes pulls each bowler toward 0 ("an average player"), but the  
          bowler value v1 has a known drift (byes / leg-byes and the real runs of wides and no-balls are not in the  
          expected runs), so the real average is below 0 and small samples are pulled up. Prior = the ball-weighted  
          mean of the metric in the data being rated.  
  BOWL-B  fix the outcome values at the root: expected team runs add, per outcome class, the average runs the fixed  
          class values miss (byes on 'dot' balls, the extra runs of wides / no-balls), learned out-of-fold by match.  
  BAT-A   the same as BOWL-A for batters (added after P9 chose v1 for batters on the corrected file: the batter v1  
          averages +0.014 runs per ball under the pressure weights, so small samples are pulled below the real average).  
  BAT-W   count dismissals off a wide: 35 striker stumpings / run-outs off a wide are currently not charged to the  
          batter (a wide is not a ball faced). They are added with the same formula as every other ball  
          (v1: bat runs 0 + run expectancy after the ball - expected; v3: the wicket component).  
Tests, as before: (i) P9's 52 unseen-match splits, (ii) the 2026 season, (iii) the outside (Cricsheet) contest of P29.  
DECISION RULE, fixed BEFORE running (symmetric; ties are reported as ties):  
  outside score = average of the two rank correlations (win chance added, runs added) on ALL public matches;  
  1,000 paired bootstraps over outside matches give P(correction beats current).  
  These are corrections of known errors, not new modelling ideas, so a tie goes to the correction:  
    P <= 0.10                -> the correction is WORSE on outside data: keep the current method.  
    0.10 < P < 0.90          -> TIE: adopt the correction (it is right by the Laws / by construction).  
    P >= 0.90                -> the correction is BETTER: adopt it.  
  The internal checks (i) and (ii) are reported next to it; if they say WORSE by more than 2 paired SEs on (i),  
  that is flagged as a warning for the user's decision.  
Sequential, 2 threads, low priority.
</details>

In [3]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# the 52 splits, yardsticks and evaluate() come from notebook 11a (run at the top of this notebook)
import sys
QUICK = "--quick" in sys.argv          # code check only: 4 splits, 20 bootstraps, nothing saved
if QUICK:
    splits = [s_ for s_ in splits if s_[0] == "rolling"] + [s_ for s_ in splits if s_[0] == "random"][:2]
from rrlib.common import load, load_json, save_table, save_json, weighted_spearman, excluded_players
try:
    import psutil
    psutil.Process().nice(psutil.BELOW_NORMAL_PRIORITY_CLASS)
except Exception:
    pass
pd.set_option("display.width", 220)

EXCL = excluded_players()
CH = load_json("p09_summary")["chosen"]
# re-test under the settings approved on 16 Nov: --bat-value v3 (submitted batting value), --bowl-k 1 (no override)
if "--bat-value" in sys.argv:
    CH["bat"] = dict(CH["bat"], value=sys.argv[sys.argv.index("--bat-value") + 1])
if "--bowl-k" in sys.argv:
    CH["bowl"] = dict(CH["bowl"], k_mult=float(sys.argv[sys.argv.index("--bowl-k") + 1]))
TAG = "_approved" if ("--bat-value" in sys.argv or "--bowl-k" in sys.argv) else ""
log.info(f"settings tested: {CH}")
R_K = np.array([0, 1, 2, 3, 4, 6, 0, 1])

# ---------------------------------------------------------------- challenger value columns
f = load("features", columns=["ball_uid", "y_class", "team_runs", "is_wide", "striker_out"])
c = load("ctx_preds", columns=["ball_uid", "fold", "p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt", "p_x"])
assert (f.ball_uid.values == V_ALL.ball_uid.values).all() and (c.ball_uid.values == V_ALL.ball_uid.values).all()
P = c[["p_dot", "p_1", "p_2", "p_3", "p_4", "p_6", "p_wkt", "p_x"]].values
miss = f.team_runs.values - R_K[f.y_class.values]          # runs the fixed class value misses on this ball
corr = np.zeros(len(f))
gap_rows = []
for k in np.unique(c.fold):
    tr, te = (c.fold != k).values, (c.fold == k).values
    gap = np.array([miss[tr & (f.y_class.values == j)].mean() for j in range(8)])
    corr[te] = (P[te] * gap[None, :]).sum(1)
    gap_rows.append({"fold": int(k), **{f"class_{j}": round(float(gap[j]), 4) for j in range(8)}})
log.info("BOWL-B missed runs per class (out-of-fold training means):\n" + pd.DataFrame(gap_rows).to_string(index=False))
V_ALL["v1b_bowl"] = V_ALL.v1_bowl + corr                   # v1_bowl = -(team + re_next - E_team); E_team += corr
V_ALL["v1b_bat"] = V_ALL.v1_bat                             # unchanged (bat runs only); needed by side_frame helpers
log.info(f"v1 bowler drift: mean v1_bowl {V_ALL.v1_bowl.mean():+.4f} -> corrected {V_ALL.v1b_bowl.mean():+.4f} runs per ball")

wd = (f.is_wide & f.striker_out).values
comp = load_json("p05_components")["bat"]
bw, rw = comp["beta"]["r_wkt_bat"], comp["reliability"]["r_wkt_bat"]
rate = wd.sum() / max(f.is_wide.sum(), 1)
V_ALL["v3w_bat"] = V_ALL.v3_bat
V_ALL.loc[wd, "v3w_bat"] = rw * bw * (1 - rate)
V_ALL["v3w_bowl"] = V_ALL.v3_bowl
# v1 version (the batting value chosen on the corrected file): the same V-1 formula as P5, bat runs 0 on a wide
c2 = load("ctx_preds", columns=["ball_uid", "re_next", "E_bat_rv"])
V_ALL["v1w_bat"] = V_ALL.v1_bat
V_ALL.loc[wd, "v1w_bat"] = (c2.re_next - c2.E_bat_rv).values[wd]
V_ALL["v1w_bowl"] = V_ALL.v1_bowl
log.info(f"BAT-W: {int(wd.sum())} dismissals off a wide added (v3: each {rw * bw * (1 - rate):+.3f}; "
         f"v1: mean {V_ALL.loc[wd, 'v1w_bat'].mean():+.2f} runs)")
# BASELINE = the method BEFORE the review fixes, so every correction is tested on its own against it (once the fixes are
# switched on in config.yaml, comparing them with the live method would compare a fix with itself):
#   shrinkage toward 0 (not the data mean), and dismissals off a wide NOT charged to the batter.
RL.SHRINK_TO = "zero"
for met in ("v1", "v3"):
    V_ALL[f"{met}base_bat"] = V_ALL[f"{met}_bat"]
    V_ALL.loc[wd, f"{met}base_bat"] = np.nan
    V_ALL[f"{met}base_bowl"] = V_ALL[f"{met}_bowl"]
log.info(f"baseline: shrink toward 0, {int(wd.sum())} dismissals off a wide removed from the batter values")
for col in ("v1b_bowl", "v1b_bat", "v3w_bat", "v3w_bowl", "v1w_bat", "v1w_bowl", "v1base_bat", "v3base_bat", "v1base_bowl", "v3base_bowl"):
    V[col] = V_ALL.loc[V_ALL.year < HOLD, col].values

# ---------------------------------------------------------------- BOWL-A: prior = real average (switchable)
_orig_shrink = RL.eb_shrink
PRIOR_MEAN = {"on": False}
def eb_shrink_real_avg(d, k_mult=1.0, prior=0.0, weight=None, key=None):
    if PRIOR_MEAN["on"]:
        prior = float(np.average(d.val, weights=None if weight is None else d[weight]))
    return _orig_shrink(d, k_mult, prior, weight, key)
RL.eb_shrink = eb_shrink_real_avg

VERS = {
    "bowl": {"current": (dict(CH["bowl"]), False),
             "BOWL-A shrink toward real average": (dict(CH["bowl"]), True),
             "BOWL-B fix outcome values": (dict(CH["bowl"], value="v1b" if CH["bowl"]["value"] == "v1" else CH["bowl"]["value"]), False)},
    "bat": {"current": (dict(CH["bat"], value=CH["bat"]["value"] + "base"), False),
            "BAT-A shrink toward real average": (dict(CH["bat"], value=CH["bat"]["value"] + "base"), True),
            "BAT-W count dismissals off a wide": (dict(CH["bat"], value={"v3": "v3w", "v1": "v1w"}.get(CH["bat"]["value"], CH["bat"]["value"])), False)},
}
# HEAD-TO-HEAD, not a correction (added 16 Nov after P9 switched the batting value v3 -> v1 on the corrected file):
# the SUBMITTED batting value v3, everything else as chosen. Rule fixed before running:
#   P(v3 beats the new pick) >= 0.90 -> CONFLICT: outside data prefers the submitted v3 -> the user decides;
#   0.10 < P < 0.90 -> TIE: the procedure's own pick stands; P <= 0.10 -> the new pick is also better outside.
if CH["bat"]["value"] != "v3":
    VERS["bat"]["BAT-V3 submitted batting value (v3)"] = (dict(CH["bat"], value="v3"), False)
H2H = {"BAT-V3 submitted batting value (v3)"}
if CH["bowl"]["value"] != "v1":
    log.info(f"NOTE: bowler value is {CH['bowl']['value']}, not v1: BOWL-B is the same as current")
if CH["bat"]["value"] not in ("v1", "v3"):
    log.info(f"NOTE: batter value is {CH['bat']['value']}: BAT-W is the same as current")
for s_ in ("bat", "bowl"):
    col_ = CH[s_]["value"] + "_" + s_
    d_ = V_ALL[V_ALL[col_].notna()]
    w_ = d_.pressure_pct.clip(1e-3, 1) ** (4 * CH[s_].get("alpha", 1.0))
    log.info(f"{s_}: chosen value {col_}; real average under the rating's pressure weights "
             f"{np.average(d_[col_], weights=w_):+.4f} runs per ball (the prior assumes 0)")


def scores(vv, side, name):
    cfg, pm = VERS[side][name]
    PRIOR_MEAN["on"] = pm
    try:
        return RL.build_scores(vv, side, cfg, comp_info).score
    finally:
        PRIOR_MEAN["on"] = False


rows = []
# (i) 52 unseen-match splits
per = {}
for side, vs in VERS.items():
    for name, (cfg, pm) in vs.items():
        PRIOR_MEAN["on"] = pm
        with Timer(log, f"(i) {side} {name}"):
            res = evaluate(cfg, (side,))
        PRIOR_MEAN["on"] = False
        per[(side, name)] = res
        r = res[res.side == side]
        rows.append({"side": side, "version": name, "test": "(i) 52 unseen-match splits",
                     "score": 0.5 * r[r.kind == "random"].rho_v2.mean() + 0.5 * r[r.kind == "rolling"].rho_v2.mean(),
                     "top10_lift_winprob_pts_per_ball": 100 * r.top10_lift_v2.mean()})
    a = per[(side, "current")].query("kind == 'random'").set_index("split").rho_v2
    for name in vs:
        if name == "current":
            continue
        dd = per[(side, name)].query("kind == 'random'").set_index("split").rho_v2.reindex(a.index) - a
        rows.append({"side": side, "version": name, "test": "(i) paired v current, 50 random splits",
                     "score": dd.mean(), "paired_se": dd.std() / np.sqrt(len(dd)), "better_share": (dd > 0).mean()})
        log.info(str(rows[-1]))

# (ii) 2026 season (fit on 2022-25)
train, test = V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]
for side, vs in VERS.items():
    y = yardstick(test, side)
    tr_ids = set(train.loc[train.is_pressure & train[f"v1_{side}"].notna(), RL.PCOL[side]].unique())
    pool = y.index[y.index.isin(tr_ids) & ~y.index.isin(EXCL)]
    yy = y.loc[pool]
    for name in vs:
        s = scores(train, side, name).reindex(pool).fillna(-1e9)
        top = s.sort_values(ascending=False).index[:10]
        rows.append({"side": side, "version": name, "test": "(ii) 2026 season", "players": len(pool),
                     "score": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                     "top10_lift_winprob_pts_per_ball": 100 * (np.average(yy.loc[top, "y2"], weights=yy.loc[top, "n"]) - np.average(yy.y2, weights=yy.n))})
        log.info(str(rows[-1]))

# (iii) outside contest (P29 yardstick and rule), full-data scores
ve = load("values_ext")
players = load("players").set_index("player_id")
FULL = {(side, name): scores(V_ALL, side, name) for side, vs in VERS.items() for name in vs}
summ = {}
for side, vs in VERS.items():
    pc = RL.PCOL[side]
    base_ids = set(V_ALL.loc[V_ALL.is_pressure & V_ALL[f"v1_{side}"].notna(), pc].unique())

    def ev(y):
        y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
        out = {}
        for name in vs:
            ss = FULL[(side, name)].reindex(y.index).fillna(-1e9)
            top = ss.sort_values(ascending=False).index[:10]
            o = {"players": len(y), "rank_corr_winprob": weighted_spearman(ss.values, y.y.values, y.n.values),
                 "rank_corr_runs": weighted_spearman(ss.values, y.y1.values, y.n.values),
                 "top10_winprob_pts_per_100": 1e4 * (np.average(y.loc[top, "y"], weights=y.loc[top, "n"]) - np.average(y.y, weights=y.n))}
            o["outside_score"] = 0.5 * (o["rank_corr_winprob"] + o["rank_corr_runs"])
            out[name] = o
        return out

    d = ve[ve.is_pressure & ve[f"wpa_{side}"].notna()]
    y = d.groupby(pc).agg(y=(f"wpa_{side}", "mean"), y1=(f"v1_{side}", "mean"), n=(f"wpa_{side}", "size"))
    allp = ev(y)
    for name, r in allp.items():
        rows.append({"side": side, "version": name, "test": "(iii) outside: all public matches", **r, "score": r["outside_score"]})
    mp = d.groupby(["match_id", pc]).agg(s=(f"wpa_{side}", "sum"), s1=(f"v1_{side}", "sum"), n=(f"wpa_{side}", "size")).reset_index()
    mids_, mi = np.unique(mp.match_id.values, return_inverse=True)
    pids, pi = np.unique(mp[pc].values, return_inverse=True)
    rng = np.random.default_rng(3101)
    boot = {k: [] for k in vs}
    with Timer(log, f"(iii) {side}: 1000 paired bootstraps over {len(mids_)} outside matches"):
        for _ in range(20 if QUICK else 1000):
            cnt = np.bincount(rng.integers(0, len(mids_), len(mids_)), minlength=len(mids_))[mi]
            n = np.bincount(pi, weights=cnt * mp.n.values, minlength=len(pids))
            ok = n > 0
            yb = pd.DataFrame({"y": np.bincount(pi, weights=cnt * mp.s.values, minlength=len(pids))[ok] / n[ok],
                               "y1": np.bincount(pi, weights=cnt * mp.s1.values, minlength=len(pids))[ok] / n[ok], "n": n[ok]}, index=pids[ok])
            for name, r in ev(yb).items():
                boot[name].append(r["outside_score"])
    Bt = pd.DataFrame(boot)
    for name in vs:
        if name == "current":
            continue
        dd = Bt[name] - Bt["current"]
        p = float((dd > 0).mean())
        i_row = [r for r in rows if r["side"] == side and r["version"] == name and r["test"].startswith("(i) paired")][0]
        warn = i_row["score"] < -2 * i_row["paired_se"]
        summ[f"{side}: {name}"] = {
            "outside_score_current": allp["current"]["outside_score"], "outside_score_correction": allp[name]["outside_score"],
            "diff": float(dd.mean()), "ci90": [float(dd.quantile(0.05)), float(dd.quantile(0.95))], "P_beats_current": p,
            "verdict": (("CONFLICT: outside data prefers the submitted v3 -> user decides" if p >= 0.90 else
                         "new pick also better outside" if p <= 0.10 else "TIE: the procedure's pick stands") if name in H2H else
                        "WORSE: keep current" if p <= 0.10 else ("BETTER: adopt" if p >= 0.90 else "TIE: adopt (correction)")),
            "internal_52_paired_diff": i_row["score"], "internal_52_paired_se": i_row["paired_se"],
            "internal_warning": bool(warn)}
    # how much the top 10 would change
    for name in vs:
        if name == "current":
            continue
        cur = FULL[(side, "current")]; alt = FULL[(side, name)]
        cur = cur[~cur.index.isin(EXCL)]; alt = alt[~alt.index.isin(EXCL)]
        t0, t1 = list(cur.index[:10]), list(alt.index[:10])
        summ[f"{side}: {name}"]["top10_changes"] = {
            "in": [str(players.name.get(p, p)) for p in t1 if p not in t0],
            "out": [str(players.name.get(p, p)) for p in t0 if p not in t1]}

R = pd.DataFrame(rows)
if not QUICK:
    save_table(R, "research_review_challengers" + TAG)
    save_json(summ, "research_review_challengers_summary" + TAG)
log.info("\n" + R.round(4).to_string(index=False))
log.info("\nDECISIONS (rule fixed in advance):\n" + pd.DataFrame(summ).T.drop(columns="top10_changes").to_string())
for k_, s_ in summ.items():
    log.info(f"{k_}: top 10 would change -> in {s_['top10_changes']['in']}, out {s_['top10_changes']['out']}")
log.info("P31 done")


19:35:41 settings tested: {'bat': {'value': 'v3', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}, 'bowl': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 60, 'alpha': 0.5}}


19:35:41 BOWL-B missed runs per class (out-of-fold training means):
 fold  class_0  class_1  class_2  class_3  class_4  class_5  class_6  class_7
    0   0.0763      0.0      0.0   0.0759      0.0      0.0   0.0284   0.3288
    1   0.0756      0.0      0.0   0.0829      0.0      0.0   0.0306   0.3280
    2   0.0756      0.0      0.0   0.0697      0.0      0.0   0.0282   0.3266
    3   0.0757      0.0      0.0   0.0720      0.0      0.0   0.0286   0.3288
    4   0.0764      0.0      0.0   0.0690      0.0      0.0   0.0282   0.3323


19:35:41 v1 bowler drift: mean v1_bowl -0.0321 -> corrected +0.0065 runs per ball


19:35:41 BAT-W: 35 dismissals off a wide added (v3: each -1.395; v1: mean -3.95 runs)


19:35:41 baseline: shrink toward 0, 35 dismissals off a wide removed from the batter values


19:35:41 bat: chosen value v3_bat; real average under the rating's pressure weights +0.0045 runs per ball (the prior assumes 0)


... (22 outputs left out)

19:36:34 bowl: BOWL-B fix outcome values: top 10 would change -> in [], out []


19:36:34 bat: BAT-A shrink toward real average: top 10 would change -> in [], out []


19:36:34 bat: BAT-W count dismissals off a wide: top 10 would change -> in [], out []


19:36:34 P31 done


## 28.2 · Batting value: v3 (kept) against v1 (the stage-by-stage pick)

<details><summary>Technical notes and the rule fixed before running</summary>

P31c - Head-to-head of the two BATTING values on the corrected file (16 Nov 2026), everything else as finally used  
(R2-weighted, alpha 0.5, shrinkage x2, floor 60, shrinkage toward the real average, dismissals off a wide charged):  
  TESTED  : v3, the value submitted on 14 Nov (boundary, dot and dismissal rates against the average batter, each  
            weighted by its run value and how repeatable it is) - kept by review decision  
  RULE_NOW: v1, the value P9's Stage A picks on the corrected file (runs added; Stage A compares values under R2-joint,  
            a family that is not used)  
Same three yardsticks as P31 / P31b. Rule fixed BEFORE running (symmetric, ties are ties):  
  P(RULE_NOW beats TESTED on the outside matches) >= 0.90 -> RULE_NOW is better outside: use it.  
  <= 0.10 -> TESTED is better outside: keep TESTED.  
  otherwise TIE -> the internal 52 checks decide: RULE_NOW only if better by more than 2 paired SEs; else keep TESTED.
</details>

In [4]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# the 52 splits, yardsticks and evaluate() come from notebook 11a (run at the top of this notebook)
from rrlib.common import load, load_json, save_table, save_json, weighted_spearman, excluded_players
EXCL = excluded_players()
players = load("players").set_index("player_id")
ch = load_json("p09_summary")["chosen"]["bat"]
SET = {"TESTED": dict(ch, value="v3"), "RULE_NOW": dict(ch, value="v1")}
log.info(f"setups: {SET}; shrink_to = {RL.SHRINK_TO}")
side, pc = "bat", "striker_id"
rows, per = [], {}
for nm, cfg in SET.items():
    per[nm] = evaluate(cfg, (side,))
    r = per[nm]
    rows.append({"setup": nm, "test": "(i) 52 splits", "score": objective(r)[side],
                 "random50": r[r.kind == "random"].rho_v2.mean(), "rolling2": r[r.kind == "rolling"].rho_v2.mean()})
dd = per["RULE_NOW"].query("kind=='random'").set_index("split").rho_v2 - per["TESTED"].query("kind=='random'").set_index("split").rho_v2
paired, pse = float(dd.mean()), float(dd.std() / np.sqrt(len(dd)))
rows.append({"setup": "RULE_NOW minus TESTED", "test": "(i) paired, 50 random", "score": paired, "paired_se": pse, "better_share": float((dd > 0).mean())})
trV, teV = V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]
y = yardstick(teV, side)
ids = set(trV.loc[trV.is_pressure & trV.v1_bat.notna(), pc].unique())
pool = y.index[y.index.isin(ids) & ~y.index.isin(EXCL)]; yy = y.loc[pool]
for nm, cfg in SET.items():
    s = RL.build_scores(trV, side, cfg, comp_info).score.reindex(pool).fillna(-1e9)
    top = s.sort_values(ascending=False).index[:10]
    rows.append({"setup": nm, "test": "(ii) 2026", "score": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                 "top10_lift_pts": 1e4 * (np.average(yy.loc[top, "y2"], weights=yy.loc[top, "n"]) - np.average(yy.y2, weights=yy.n))})
FULL = {nm: RL.build_scores(V_ALL, side, cfg, comp_info).score for nm, cfg in SET.items()}
ve = load("values_ext")
base_ids = set(V_ALL.loc[V_ALL.is_pressure & V_ALL.v1_bat.notna(), pc].unique())


def ev(y):
    y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
    out = {}
    for nm in SET:
        ss = FULL[nm].reindex(y.index).fillna(-1e9); top = ss.sort_values(ascending=False).index[:10]
        o = {"players": len(y), "rank_corr_winprob": weighted_spearman(ss.values, y.y.values, y.n.values),
             "rank_corr_runs": weighted_spearman(ss.values, y.y1.values, y.n.values),
             "top10_winprob_pts_per_100": 1e4 * (np.average(y.loc[top, "y"], weights=y.loc[top, "n"]) - np.average(y.y, weights=y.n))}
        o["outside_score"] = 0.5 * (o["rank_corr_winprob"] + o["rank_corr_runs"]); out[nm] = o
    return out


d = ve[ve.is_pressure & ve.wpa_bat.notna()]
allp = ev(d.groupby(pc).agg(y=("wpa_bat", "mean"), y1=("v1_bat", "mean"), n=("wpa_bat", "size")))
for nm, r in allp.items():
    rows.append({"setup": nm, "test": "(iii) outside, all public", **r, "score": r["outside_score"]})
mp = d.groupby(["match_id", pc]).agg(s=("wpa_bat", "sum"), s1=("v1_bat", "sum"), n=("wpa_bat", "size")).reset_index()
mids_, mi = np.unique(mp.match_id.values, return_inverse=True); pids, pi = np.unique(mp[pc].values, return_inverse=True)
rng = np.random.default_rng(3103); boot = {k: [] for k in SET}
for _ in range(1000):
    cnt = np.bincount(rng.integers(0, len(mids_), len(mids_)), minlength=len(mids_))[mi]
    n = np.bincount(pi, weights=cnt * mp.n.values, minlength=len(pids)); ok = n > 0
    yb = pd.DataFrame({"y": np.bincount(pi, weights=cnt * mp.s.values, minlength=len(pids))[ok] / n[ok],
                       "y1": np.bincount(pi, weights=cnt * mp.s1.values, minlength=len(pids))[ok] / n[ok], "n": n[ok]}, index=pids[ok])
    for nm, r in ev(yb).items():
        boot[nm].append(r["outside_score"])
Bt = pd.DataFrame(boot); dd3 = Bt.RULE_NOW - Bt.TESTED; P = float((dd3 > 0).mean())
if P >= 0.90:
    verdict = "RULE_NOW better outside: use it"
elif P <= 0.10:
    verdict = "TESTED better outside: keep TESTED"
else:
    verdict = ("TIE outside; RULE_NOW clears the higher internal bar (> 2 SE): use it" if paired > 2 * pse
               else "TIE outside; RULE_NOW does not clear the internal bar (> 2 SE better): keep TESTED")
lists = {}
for nm in SET:
    s = FULL[nm]; s = s[~s.index.isin(EXCL)]
    lists[nm] = [str(players.name.get(i, i)) for i in s.index[:12]]
summ = {"setups": SET, "outside_P_rule_now_beats_tested": P, "outside_diff": float(dd3.mean()),
        "outside_ci90": [float(dd3.quantile(.05)), float(dd3.quantile(.95))], "internal_paired": paired, "internal_se": pse,
        "verdict": verdict, "top12": lists}
R = pd.DataFrame(rows)
save_table(R, "research_batting_value_h2h"); save_json(summ, "research_batting_value_h2h")
log.info("\n" + R.round(4).to_string(index=False))
log.info(f"outside P(RULE_NOW beats TESTED) = {P:.2f}; internal paired {paired:+.4f} (SE {pse:.4f}) -> {verdict}")
for nm, l in lists.items():
    log.info(f"{nm} top 12: {l}")
log.info("P31c done")


19:36:34 setups: {'TESTED': {'value': 'v3', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}, 'RULE_NOW': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 2.0, 'min_n': 60, 'alpha': 0.5}}; shrink_to = zero


19:36:52 
                setup                      test   score  random50  rolling2  paired_se  better_share  top10_lift_pts  players  rank_corr_winprob  rank_corr_runs  top10_winprob_pts_per_100  outside_score
               TESTED             (i) 52 splits  0.1269    0.1433    0.1104        NaN           NaN             NaN      NaN                NaN             NaN                        NaN            NaN
             RULE_NOW             (i) 52 splits  0.1196    0.1355    0.1037        NaN           NaN             NaN      NaN                NaN             NaN                        NaN            NaN
RULE_NOW minus TESTED     (i) paired, 50 random -0.0079       NaN       NaN     0.0026          0.36             NaN      NaN                NaN             NaN                        NaN            NaN
               TESTED                 (ii) 2026 -0.0528       NaN       NaN        NaN           NaN        -22.8792      NaN                NaN             NaN                  

19:36:52 outside P(RULE_NOW beats TESTED) = 0.23; internal paired -0.0079 (SE 0.0026) -> TIE outside; RULE_NOW does not clear the internal bar (> 2 SE better): keep TESTED


19:36:52 TESTED top 12: ['Abhishek Sharma', 'Vaibhav Sooryavanshi', 'Suryakumar Yadav', 'Phil Salt', 'Tim David', 'Yashasvi Jaiswal', 'Heinrich Klaasen', 'Josh Inglis', 'Rovman Powell', 'Priyansh Arya', 'Travis Head', 'Nicholas Pooran']


19:36:52 RULE_NOW top 12: ['Abhishek Sharma', 'Suryakumar Yadav', 'Tim David', 'Vaibhav Sooryavanshi', 'Heinrich Klaasen', 'Rajat Patidar', 'Nicholas Pooran', 'Phil Salt', 'Rovman Powell', 'Virat Kohli', 'Yashasvi Jaiswal', 'Jos Buttler']


19:36:52 P31c done


## 28.3 · Bowler setup: the tested setup against the drifted rule pick

<details><summary>Technical notes and the rule fixed before running</summary>

P31b - Head-to-head of two complete BOWLER setups on the corrected file, with shrinkage toward the real average  
(config ratings.shrink_to = data_mean) switched on for both:  
  TESTED  : v1, R2-weighted, alpha 0.5, shrinkage x1, no consistency weight, floor 60 (the setup the P31 corrections  
            were tested on, and the rule's own pick before the shrink-target fix)  
  RULE_NOW: what P9's stage-by-stage search picks once the shrink-target fix is on (read from p09_summary 'rule_pick',  
            with the 60-ball floor)  
Same three yardsticks as P31. Rule fixed BEFORE running (symmetric, ties are ties):  
  P(RULE_NOW beats TESTED on the outside matches) >= 0.90 -> RULE_NOW is better outside: use it.  
  <= 0.10 -> TESTED is better outside: keep TESTED and record why the rule's pick was not used.  
  otherwise TIE -> the internal 52 checks decide (paired, 50 random splits): RULE_NOW only if it is better by more than  
  2 paired SEs (it was tuned on those checks, so it must clear a higher bar); otherwise keep TESTED (simpler: no  
  consistency term) and report the tie.
</details>

In [5]:
import os
os.environ["OMP_NUM_THREADS"] = "2"
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

# the 52 splits, yardsticks and evaluate() come from notebook 11a (run at the top of this notebook)
from rrlib.common import load, load_json, save_table, save_json, weighted_spearman, excluded_players
EXCL = excluded_players()
players = load("players").set_index("player_id")
rp = load_json("p09_summary")["rule_pick"]["bowl"]
SET = {"TESTED": {"value": "v1", "model": "R2-weighted", "w_d": 0.0, "w_c": 0.0, "k_mult": 1.0, "min_n": 60, "alpha": 0.5},
       "RULE_NOW": dict(rp, min_n=60)}
log.info(f"setups: {SET}; shrink_to = {RL.SHRINK_TO}")
side, pc = "bowl", "bowler_id"
rows, per = [], {}
for nm, cfg in SET.items():
    per[nm] = evaluate(cfg, (side,))
    r = per[nm]
    rows.append({"setup": nm, "test": "(i) 52 splits", "score": objective(r)[side],
                 "random50": r[r.kind == "random"].rho_v2.mean(), "rolling2": r[r.kind == "rolling"].rho_v2.mean()})
dd = per["RULE_NOW"].query("kind=='random'").set_index("split").rho_v2 - per["TESTED"].query("kind=='random'").set_index("split").rho_v2
paired, pse = float(dd.mean()), float(dd.std() / np.sqrt(len(dd)))
rows.append({"setup": "RULE_NOW minus TESTED", "test": "(i) paired, 50 random", "score": paired, "paired_se": pse, "better_share": float((dd > 0).mean())})
trV, teV = V_ALL[V_ALL.year < HOLD], V_ALL[V_ALL.year == HOLD]
y = yardstick(teV, side)
ids = set(trV.loc[trV.is_pressure & trV.v1_bowl.notna(), pc].unique())
pool = y.index[y.index.isin(ids) & ~y.index.isin(EXCL)]; yy = y.loc[pool]
for nm, cfg in SET.items():
    s = RL.build_scores(trV, side, cfg, comp_info).score.reindex(pool).fillna(-1e9)
    top = s.sort_values(ascending=False).index[:10]
    rows.append({"setup": nm, "test": "(ii) 2026", "score": weighted_spearman(s.values, yy.y2.values, yy.n.values),
                 "top10_lift_pts": 1e4 * (np.average(yy.loc[top, "y2"], weights=yy.loc[top, "n"]) - np.average(yy.y2, weights=yy.n))})
FULL = {nm: RL.build_scores(V_ALL, side, cfg, comp_info).score for nm, cfg in SET.items()}
ve = load("values_ext")
base_ids = set(V_ALL.loc[V_ALL.is_pressure & V_ALL.v1_bowl.notna(), pc].unique())


def ev(y):
    y = y[(y.n >= 20) & y.index.isin(players.index) & y.index.isin(base_ids) & ~y.index.isin(EXCL)]
    out = {}
    for nm in SET:
        ss = FULL[nm].reindex(y.index).fillna(-1e9); top = ss.sort_values(ascending=False).index[:10]
        o = {"players": len(y), "rank_corr_winprob": weighted_spearman(ss.values, y.y.values, y.n.values),
             "rank_corr_runs": weighted_spearman(ss.values, y.y1.values, y.n.values),
             "top10_winprob_pts_per_100": 1e4 * (np.average(y.loc[top, "y"], weights=y.loc[top, "n"]) - np.average(y.y, weights=y.n))}
        o["outside_score"] = 0.5 * (o["rank_corr_winprob"] + o["rank_corr_runs"]); out[nm] = o
    return out


d = ve[ve.is_pressure & ve.wpa_bowl.notna()]
allp = ev(d.groupby(pc).agg(y=("wpa_bowl", "mean"), y1=("v1_bowl", "mean"), n=("wpa_bowl", "size")))
for nm, r in allp.items():
    rows.append({"setup": nm, "test": "(iii) outside, all public", **r, "score": r["outside_score"]})
mp = d.groupby(["match_id", pc]).agg(s=("wpa_bowl", "sum"), s1=("v1_bowl", "sum"), n=("wpa_bowl", "size")).reset_index()
mids_, mi = np.unique(mp.match_id.values, return_inverse=True); pids, pi = np.unique(mp[pc].values, return_inverse=True)
rng = np.random.default_rng(3102); boot = {k: [] for k in SET}
for _ in range(1000):
    cnt = np.bincount(rng.integers(0, len(mids_), len(mids_)), minlength=len(mids_))[mi]
    n = np.bincount(pi, weights=cnt * mp.n.values, minlength=len(pids)); ok = n > 0
    yb = pd.DataFrame({"y": np.bincount(pi, weights=cnt * mp.s.values, minlength=len(pids))[ok] / n[ok],
                       "y1": np.bincount(pi, weights=cnt * mp.s1.values, minlength=len(pids))[ok] / n[ok], "n": n[ok]}, index=pids[ok])
    for nm, r in ev(yb).items():
        boot[nm].append(r["outside_score"])
Bt = pd.DataFrame(boot); dd3 = Bt.RULE_NOW - Bt.TESTED; P = float((dd3 > 0).mean())
if P >= 0.90:
    verdict = "RULE_NOW better outside: use it"
elif P <= 0.10:
    verdict = "TESTED better outside: keep TESTED"
else:
    verdict = ("TIE outside; RULE_NOW clears the higher internal bar (> 2 SE): use it" if paired > 2 * pse
               else "TIE outside; internal gain not > 2 SE: keep TESTED (simpler)")
lists = {}
for nm in SET:
    s = FULL[nm]; s = s[~s.index.isin(EXCL)]
    lists[nm] = [str(players.name.get(i, i)) for i in s.index[:12]]
summ = {"setups": SET, "outside_P_rule_now_beats_tested": P, "outside_diff": float(dd3.mean()),
        "outside_ci90": [float(dd3.quantile(.05)), float(dd3.quantile(.95))], "internal_paired": paired, "internal_se": pse,
        "verdict": verdict, "top12": lists}
R = pd.DataFrame(rows)
save_table(R, "research_bowler_setup_h2h"); save_json(summ, "research_bowler_setup_h2h")
log.info("\n" + R.round(4).to_string(index=False))
log.info(f"outside P(RULE_NOW beats TESTED) = {P:.2f}; internal paired {paired:+.4f} (SE {pse:.4f}) -> {verdict}")
for nm, l in lists.items():
    log.info(f"{nm} top 12: {l}")
log.info("P31b done")


19:36:52 setups: {'TESTED': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 1.0, 'min_n': 60, 'alpha': 0.5}, 'RULE_NOW': {'value': 'v1', 'model': 'R2-weighted', 'w_d': 0.0, 'w_c': 0.0, 'k_mult': 0.5, 'min_n': 60, 'alpha': 0.5}}; shrink_to = zero


19:37:11 
                setup                      test   score  random50  rolling2  paired_se  better_share  top10_lift_pts  players  rank_corr_winprob  rank_corr_runs  top10_winprob_pts_per_100  outside_score
               TESTED             (i) 52 splits  0.1013    0.1096    0.0929        NaN           NaN             NaN      NaN                NaN             NaN                        NaN            NaN
             RULE_NOW             (i) 52 splits  0.1015    0.1096    0.0935        NaN           NaN             NaN      NaN                NaN             NaN                        NaN            NaN
RULE_NOW minus TESTED     (i) paired, 50 random -0.0000       NaN       NaN     0.0002           0.5             NaN      NaN                NaN             NaN                        NaN            NaN
               TESTED                 (ii) 2026  0.1646       NaN       NaN        NaN           NaN         53.9572      NaN                NaN             NaN                  

19:37:11 outside P(RULE_NOW beats TESTED) = 0.70; internal paired -0.0000 (SE 0.0002) -> TIE outside; internal gain not > 2 SE: keep TESTED (simpler)


19:37:11 TESTED top 12: ['Jasprit Bumrah', 'Lungi Ngidi', 'Benjamin Sears', 'Mitchell Santner', 'Rashid Khan', 'Josh Hazlewood', 'Sunil Narine', 'Brydon Carse', 'Nathan Ellis', 'Jamie Overton', 'Ish Sodhi', 'Taskin Ahmed']


19:37:11 RULE_NOW top 12: ['Jasprit Bumrah', 'Benjamin Sears', 'Lungi Ngidi', 'Brydon Carse', 'Mitchell Santner', 'Sunil Narine', 'Josh Hazlewood', 'Jamie Overton', 'Nathan Ellis', 'Rashid Khan', 'Ish Sodhi', 'Ottneil Baartman']


19:37:11 P31b done
